# Posture Detection V2 — 06 Real-Time MotionBERT Webcam

Live MediaPipe → 81-frame buffer → asynchronous MotionBERT → personal calibration → score/issues/alerts. Press **Q** to quit and **C** to recalibrate. Sit upright during calibration. CPU inference updates periodically while the camera remains responsive.

## 1. Setup

In [1]:
import sys, time, json
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
import cv2, mediapipe as mp, numpy as np, pandas as pd, torch

wd=Path.cwd(); PROJECT_ROOT=wd.parent if wd.name=='notebooks' else wd if wd.name=='project_v2' else wd/'project_v2'
MB_ROOT=PROJECT_ROOT/'motionbert'/'MotionBERT'; MODEL_TASK=PROJECT_ROOT/'models'/'pose_landmarker_full.task'
CONFIG=MB_ROOT/'configs/pose3d/MB_ft_h36m_global_lite.yaml'
CHECKPOINT=MB_ROOT/'checkpoint/pose3d/FT_MB_lite_MB_ft_h36m_global_lite/best_epoch.bin'
OUTPUT_DIR=PROJECT_ROOT/'results'/'webcam_sessions'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for p in (MB_ROOT,MODEL_TASK,CONFIG,CHECKPOINT):
    if not p.exists(): raise FileNotFoundError(f'Missing {p}; complete Notebooks 02 and 03 first.')
sys.path.insert(0,str(MB_ROOT.resolve()))
from lib.utils.learning import load_backbone
from lib.utils.tools import get_config
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); args=get_config(str(CONFIG)); model=load_backbone(args)
state=torch.load(CHECKPOINT,map_location='cpu',weights_only=False)['model_pos']
try: model.load_state_dict(state,strict=True)
except RuntimeError: model.load_state_dict({k.removeprefix('module.'):v for k,v in state.items()},strict=True)
model=model.to(device).eval(); print('Device:',device)

Device: cpu


## 2. Mapping, 3D lifting, and calibrated scoring

In [2]:
MP={'nose':0,'le':7,'re':8,'ls':11,'rs':12,'lelb':13,'relb':14,'lw':15,'rw':16,'lh':23,'rh':24,'lk':25,'rk':26,'la':27,'ra':28}
def avg(a,b): return np.nanmean(np.stack([a,b]),axis=0)
def to_h36m(landmarks):
    a=np.array([[p.x,p.y,min(p.visibility,getattr(p,'presence',p.visibility))] for p in landmarks],np.float32); o=np.empty((17,3),np.float32)
    pelvis=avg(a[MP['lh']],a[MP['rh']]); thorax=avg(a[MP['ls']],a[MP['rs']]); head=avg(a[MP['le']],a[MP['re']])
    o[:]=[pelvis,a[MP['rh']],a[MP['rk']],a[MP['ra']],a[MP['lh']],a[MP['lk']],a[MP['la']],avg(pelvis,thorax),thorax,a[MP['nose']],head,a[MP['ls']],a[MP['lelb']],a[MP['lw']],a[MP['rs']],a[MP['relb']],a[MP['rw']]]; return o
def prepare(window,w,h):
    x=window.copy(); x[...,0]=x[...,0]*2-1; x[...,1]=(x[...,1]*2-1)*(h/w); x[...,2]=np.clip(x[...,2],0,1); return x
def lift(window):
    started=time.perf_counter(); x=torch.from_numpy(window[None]).to(device)
    with torch.inference_mode(): y=model(x)
    y=y.cpu().numpy()[0]; y-=y[:,0:1]; return y[-1],(time.perf_counter()-started)*1000
def features(p):
    pelvis,thorax,head,ls,rs=p[0],p[8],p[10],p[11],p[14]; torso=thorax-pelvis; neck=head-thorax; shoulder=ls-rs
    tl=max(np.linalg.norm(torso),1e-6); sw=max(np.linalg.norm(shoulder),1e-6)
    angle=np.degrees(np.arccos(np.clip(np.dot(neck,torso)/(max(np.linalg.norm(neck),1e-6)*tl),-1,1)))
    return np.array([neck[2]/tl,angle,torso[0]/tl,torso[2]/tl,(ls[1]-rs[1])/sw],float)
FLOOR=np.array([.08,8,.06,.08,.05]); NAMES=np.array(['forward head','slouching','leaning','slouching','shoulder imbalance'])
def score_pose(value,baseline,scale):
    sev=np.clip((np.abs(value-baseline)/scale-1)/2,0,1); grouped=np.array([sev[0],max(sev[1],sev[3]),sev[2],sev[4]])
    score=float(np.clip(100*(1-np.dot(grouped,[.30,.35,.20,.15])),0,100)); i=int(np.argmax(grouped)); issue=np.array(['forward head','slouching','leaning','shoulder imbalance'])[i] if grouped[i]>=.5 else 'none'
    return score,issue,float(grouped[i])

## 3. Run webcam

Calibration begins after the first 81 frames and requires 8 successful 3D updates. Keep head, shoulders, hips and preferably knees visible. The history CSV is saved when you quit.

In [3]:
CAMERA_INDEX=0; WINDOW=81; STRIDE=27; CALIBRATION_UPDATES=8; BAD_THRESHOLD=70; ALERT_AFTER=5; COOLDOWN=30
Base=mp.tasks.BaseOptions; Landmarker=mp.tasks.vision.PoseLandmarker; Options=mp.tasks.vision.PoseLandmarkerOptions; Mode=mp.tasks.vision.RunningMode
options=Options(base_options=Base(model_asset_path=str(MODEL_TASK)),running_mode=Mode.VIDEO,num_poses=1,min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,min_tracking_confidence=.5)
cap=cv2.VideoCapture(CAMERA_INDEX); cap.set(cv2.CAP_PROP_FRAME_WIDTH,1280); cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
if not cap.isOpened(): raise RuntimeError('Could not open webcam.')
buffer=deque(maxlen=WINDOW); executor=ThreadPoolExecutor(max_workers=1); future=None; calibration=[]; baseline=scale=None
score=issue_text=None; severity=0.; latency=np.nan; bad_since=None; last_alert=-np.inf; history=[]; frame_id=0; started=time.perf_counter()
with Landmarker.create_from_options(options) as detector:
  try:
    while True:
      ok,frame=cap.read();
      if not ok: break
      frame=cv2.flip(frame,1); h,w=frame.shape[:2]; now=time.perf_counter(); timestamp=int((now-started)*1000)
      result=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)),timestamp)
      detected=bool(result.pose_landmarks)
      if detected: buffer.append(to_h36m(result.pose_landmarks[0]))
      else: buffer.clear()
      if future is not None and future.done():
        try:
          pose3d,latency=future.result(); value=features(pose3d)
          if baseline is None:
            calibration.append(value); status=f'CALIBRATING {len(calibration)}/{CALIBRATION_UPDATES}'
            if len(calibration)>=CALIBRATION_UPDATES:
              a=np.stack(calibration); baseline=np.median(a,axis=0); scale=np.maximum(1.4826*np.median(np.abs(a-baseline),axis=0),FLOOR); status='CALIBRATED'
          else:
            score,issue_text,severity=score_pose(value,baseline,scale); bad=score<BAD_THRESHOLD
            bad_since=bad_since or now if bad else None; duration=0 if bad_since is None else now-bad_since
            alert=duration>=ALERT_AFTER and now-last_alert>=COOLDOWN
            if alert: last_alert=now
            status='ALERT' if alert else 'MONITORING BAD' if bad else 'GOOD'
            history.append({'utc':datetime.now(timezone.utc).isoformat(),'seconds':now-started,'score':score,'issue':issue_text,'severity':severity,'state':status,'bad_duration_seconds':duration,'latency_ms':latency})
        except Exception as e: status=f'INFERENCE ERROR: {e}'
        future=None
      if len(buffer)==WINDOW and frame_id%STRIDE==0 and future is None: future=executor.submit(lift,prepare(np.stack(buffer),w,h))
      if baseline is None: status=locals().get('status',f'BUFFERING {len(buffer)}/{WINDOW}')
      color=(0,220,0) if status in ('GOOD','CALIBRATED') else (0,200,255) if 'CALIB' in status or 'BUFFER' in status else (0,0,255)
      cv2.putText(frame,status,(25,40),cv2.FONT_HERSHEY_SIMPLEX,.9,color,2)
      if score is not None:
        cv2.putText(frame,f'Score: {score:5.1f}  Issue: {issue_text}',(25,80),cv2.FONT_HERSHEY_SIMPLEX,.75,color,2); cv2.putText(frame,f'MotionBERT: {latency:.0f} ms',(25,115),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),2)
      cv2.putText(frame,'Q quit | C recalibrate',(25,h-25),cv2.FONT_HERSHEY_SIMPLEX,.55,(255,255,255),1); cv2.imshow('Posture V2 - MotionBERT',frame)
      key=cv2.waitKey(1)&0xFF
      if key==ord('q'): break
      if key==ord('c'): calibration=[]; baseline=scale=None; score=issue_text=None; bad_since=None; status='RECALIBRATING'
      frame_id+=1
  finally:
    cap.release(); cv2.destroyAllWindows(); executor.shutdown(wait=True)
session=pd.DataFrame(history); path=OUTPUT_DIR/f"webcam_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"; session.to_csv(path,index=False)
print('Saved:',path.resolve()); display(session.tail())

Saved: C:\Users\manth\Desktop\project_v2\results\webcam_sessions\webcam_session_20260923_015031.csv


,utc,seconds,score,issue,severity,state,bad_duration_seconds,latency_ms
28,2026-09-22T20:20:24.182865+00:00,56.582228,88.513856,shoulder imbalance,0.765743,GOOD,0.0,1220.1342
29,2026-09-22T20:20:25.843708+00:00,58.248065,87.236356,shoulder imbalance,0.850910,GOOD,0.0,1332.7936
30,2026-09-22T20:20:27.266177+00:00,59.668393,91.848925,shoulder imbalance,0.543405,GOOD,0.0,1196.0242
31,2026-09-22T20:20:28.777251+00:00,61.168816,63.149649,shoulder imbalance,1.000000,MONITORING BAD,0.0,1213.9085
32,2026-09-22T20:20:30.240784+00:00,62.634017,85.494201,shoulder imbalance,0.967053,GOOD,0.0,1159.0140


## Interpretation

This is near-real-time on CPU: the display remains live, while the posture result updates every 27 frames after asynchronous inference. Public MultiPosture training can later replace the `score_pose` rule function through a compatible classifier adapter; it should be evaluated separately because its 11-joint skeleton and label set do not match MotionBERT exactly.